<a href="https://colab.research.google.com/github/alireza1420/stargazer-prediction-pipeline/blob/carolines-neural-net/Best_Model_May_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
import joblib

class GitHubRepoPreprocessor:
    def __init__(self, reference_date=None):
        self.date = reference_date or datetime(2025, 5, 1)
        self.numeric_features = [
            'forks', 'open_issues', 'size', 'subscribers_count',
            'contributors_count', 'commits_count', 'readme_size',
            'project_age', 'days_since_update', 'days_since_push',
            'forks_per_day', 'issues_per_day', 'update_rate'
        ]
        self.categorical_features = ["language", "license"]
        self.column_transformer = None

    def transform(self, df):
        df["created_at"] = pd.to_datetime(df["created_at"]).dt.tz_localize(None)
        df["updated_at"] = pd.to_datetime(df["updated_at"]).dt.tz_localize(None)
        df["pushed_at"] = pd.to_datetime(df["pushed_at"]).dt.tz_localize(None)

        df["project_age"] = (self.date - df["created_at"]).dt.days
        df["days_since_update"] = (self.date - df["updated_at"]).dt.days
        df["days_since_push"] = (self.date - df["pushed_at"]).dt.days

        df["license"] = df["license"].fillna("None")
        df["language"] = df["language"].fillna("Unknown")

        df["forks_per_day"] = df["forks"] / (df["project_age"] + 1)
        df["issues_per_day"] = df["open_issues"] / (df["project_age"] + 1)
        df["update_rate"] = 1 / (1 + df["days_since_update"])

        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.dropna(inplace=True)

        df["has_wiki"] = df["has_wiki"].astype(int)
        df["has_projects"] = df["has_projects"].astype(int)
        df["has_downloads"] = df["has_downloads"].astype(int)
        df["is_fork"] = df["is_fork"].astype(int)
        df["archived"] = df["archived"].astype(int)

        selected_features = [
            'forks', 'open_issues', 'size', 'has_wiki', 'has_projects', 'has_downloads',
            'is_fork', 'archived', 'language', 'license',
            'subscribers_count', 'contributors_count', 'commits_count', 'readme_size',
            'project_age', 'days_since_update', 'days_since_push',
            'forks_per_day', 'issues_per_day', 'update_rate'
        ]

        return df[selected_features], df["stars"]

    def get_preprocessor(self):
        self.column_transformer = ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), self.numeric_features),
                ("cat", OneHotEncoder(handle_unknown="ignore"), self.categorical_features)
            ],
            remainder='passthrough'
        )
        return self.column_transformer


df = pd.read_csv('github_repo_features.csv')
preprocessor = GitHubRepoPreprocessor()
X, y = preprocessor.transform(df)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor_pipeline = preprocessor.get_preprocessor()
X_train_transformed = preprocessor_pipeline.fit_transform(X_train)
X_test_transformed = preprocessor_pipeline.transform(X_test)


#define the model
model = Sequential()
model.add(tf.keras.Input(shape=(X_train_transformed.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1))

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='mse', optimizer=optimizer, metrics=['mae'])

#fit the model to the dataset
model.fit(X_train_transformed, y_train, epochs=150, batch_size=10, validation_split=0.2)

model.save('neural_network_new_model.h5')

# For prediction and evaluation, also use the transformed test data
predictions = model.predict(X_test_transformed)

#evaluate the model
_, R2_score = model.evaluate(X_test_transformed, y_test)
print('R2 score:', r2_score(y_test, predictions))

Epoch 1/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 4766529536.0000 - mae: 52239.2148 - val_loss: 4651357696.0000 - val_mae: 53579.0195
Epoch 2/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 5006055424.0000 - mae: 52376.4102 - val_loss: 4649741312.0000 - val_mae: 53564.8008
Epoch 3/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 4676747264.0000 - mae: 52229.9492 - val_loss: 4643484160.0000 - val_mae: 53510.5859
Epoch 4/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 4130416384.0000 - mae: 49920.3672 - val_loss: 4627695104.0000 - val_mae: 53374.9492
Epoch 5/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 4563963392.0000 - mae: 52269.8867 - val_loss: 4598458368.0000 - val_mae: 53124.0078
Epoch 6/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 4846243328.0000 - mae: 52845.4258 - val_loss: 4552375808.0000 - val_mae: 52728.4141
Epoch 7/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 4707031552.0000 - mae: 51312.6562 - val_loss: 4487165440.0000 - val_mae: 52162.1016

1/7 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 382705568.0000 - mae: 13320.7412 
R2 score: 0.4807802438735962


In [15]:
# Create a real-world sample repository
sample_repo = {
    'name': 'ml-web-app',
    'full_name': 'data-scientist/ml-web-app',
    'created_at': '2023-05-10T08:00:00Z',
    'updated_at': '2023-11-15T14:25:00Z',
    'pushed_at': '2023-11-15T14:30:00Z',
    'language': 'Python',  # Must be in encoders['language'].classes_
    'license': 'mit',      # Must be in encoders['license'].classes_
    'forks': 87,
    'watchers': 420, # same as stars, unknown
    'open_issues': 12,
    'size': 3500,
    'has_wiki': True,
    'has_projects': False,
    'has_downloads': True,
    'is_fork': False,
    'archived': False,
    'subscribers_count': 150,
    'readme_size': 1024,
    'commits_count': 85,
    'contributors_count': 12
}


# Load the Keras model
loaded_model = tf.keras.models.load_model("neural_network_new_model.h5", custom_objects={'mse': tf.keras.losses.MeanSquaredError()})

sample_df = pd.DataFrame([sample_repo])

sample_df.rename(columns={'watchers': 'stars'}, inplace=True)

sample_processed, _ = preprocessor.transform(sample_df)

sample_transformed = preprocessor_pipeline.transform(sample_processed)

predicted_stars = loaded_model.predict(sample_transformed)

print("Predicted number of stars for sample repo:", predicted_stars[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
Predicted number of stars for sample repo: 629552.2
